# Openrouter model

In [15]:
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy
load_dotenv()

True

In [11]:
def get_weather(city: str) -> str:
    """" Get weather for a given city """
    return f"The weather in {city} is sunny"

model= ChatOpenRouter( model="arcee-ai/trinity-large-preview:free")

agent= create_agent(model=model, tools=[get_weather], system_prompt="You are a helpful agent")
agent.invoke({"messages": [{"role": "user", "content": "what is the weather in sf"}]})

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='ffe46a7e-d7f1-4863-a993-4ad7c6b4a5c5'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'arcee-ai/trinity-large-preview:free', 'id': 'gen-1772101514-RTfeGLz2veOchMLzwP0O', 'created': 1772101514, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019c997b-1f58-7130-a93b-cf96eeca25b3-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call-8e12f957-85db-4078-a204-e7115eba0632', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_tokens': 21, 'total_tokens': 162, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}),
  ToolMessage(content='The weather in San Francisco is sunny', name='get_weather', id='273658e1-b6cb-450e-a16b-7c08bc9d5e45', tool_call_id='call-8e12f957-85db-4078-a

In [60]:
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. 
If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

@tool
def get_weather_for_location(location: str) -> str:
    """" Get weather for a given location. Assume exact location if necessary. """
    return f"The weather in {location} is sunny"

@dataclass
class Context:
    """" Custom runtime context schema. """
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """" Get the user's location based on user ID. """
    user_id= runtime.context.user_id
    return "Florida" if user_id=="1" else "New York"

model= ChatOpenRouter( model="stepfun/step-3.5-flash:free", temperature= 1, max_tokens=1000)

@dataclass
class ResponseFormat:
    """" Response schema for the agent. """
    # It always need to be punny (Required)
    punny_response: str
    #Any interesting information about the weather if available
    weather_conditions: str|None= None

checkpointer= InMemorySaver()

In [ ]:
agent= create_agent(
    model= model, 
    system_prompt= SYSTEM_PROMPT, 
    tools=[get_user_location, get_weather_for_location], 
    context_schema= Context, 
    response_format= ResponseFormat,
    checkpointer= checkpointer
    )

config= {"configurable": {"thread_id": "1"}}

response= agent.invoke(
    {"messages": [{"role":"user", "content": "What is the weather outside?"}]},
    config= config,
    context=Context(user_id= "1")
)

print(response['structured_response'])
# print(response)

ResponseFormat(punny_response="It's sunny in Florida, so you can really soak up the rays-itude!", weather_conditions='sunny')


In [69]:
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1") #  same user id allows to continue the conversation
)

# print(response1['structured_response'])
print(response1['messages'][-1].content)

You're as refreshing as a rain shower in summer! Let me know if you need any more weather wonders! 🌦️


In [63]:
response1

{'messages': [HumanMessage(content='What is the weather ooutside?', additional_kwargs={}, response_metadata={}, id='0db60889-e0e1-48f2-81cd-08e673a662df'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking about the weather "outside" but hasn\'t specified a location. Since they said "outside" which naturally means where they are currently located, I should get their location using the get_user_location tool first. Then I can get the weather for that location.\n\nLet me call get_user_location to find out where they are.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user is asking about the weather "outside" but hasn\'t specified a location. Since they said "outside" which naturally means where they are currently located, I should get their location using the get_user_location tool first. Then I can get the weather for that location.\n\nLet me call get_user_location to find out where they are.', 'format': 'unknown', 'index': 0.0}]}, res

# Mistral model

In [46]:
from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain.agents import create_agent
from langchain.tools import tool
from tavily import TavilyClient
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()

True

In [2]:
model = ChatMistralAI(
    model="mistral-medium-2508",
)

## Simple tool call

In [46]:
@tool("Square_root", description="Get the square root of a number.")
def tool1(x: float) -> float:
    # """" Get the square root of a number. """
    return x ** 0.5

In [49]:
agent= create_agent(
    model= model,
    tools=[tool1],
    system_prompt="You are a helpful agent"
)

In [54]:
input= {'messages': [{'role': 'user', 'content': 'What is the square root of 224?'}]}
response= agent.invoke(input)
print(response['messages'][-1].content)


The square root of **224** is approximately **14.9666**.


In [56]:
response['messages']

[HumanMessage(content='What is the square root of 224?', additional_kwargs={}, response_metadata={}, id='bc2ddbdd-9da9-4fc5-930f-5d01fdfba22f'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'LDKSwWuuc', 'function': {'name': 'Square_root', 'arguments': '{"x": 224}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 81, 'total_tokens': 94, 'completion_tokens': 13, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-2508', 'model': 'mistral-medium-2508', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019c9dd7-0c39-7da3-949d-0f4c69e9e379-0', tool_calls=[{'name': 'Square_root', 'args': {'x': 224}, 'id': 'LDKSwWuuc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 81, 'output_tokens': 13, 'total_tokens': 94}),
 ToolMessage(content='14.966629547095765', name='Square_root', id='c6bdc473-3c1e-4456-8865-a1d5fdb46273', tool_call_id='LDKSwWuuc'),
 AIMessage(content='The square r

In [57]:
response['messages'][1].tool_calls

[{'name': 'Square_root',
  'args': {'x': 224},
  'id': 'LDKSwWuuc',
  'type': 'tool_call'}]

## Web search tool

In [63]:
agent= create_agent(
    model= model
)
Question= "How up to date is your training knowledge?"
input={'messages': [{'role': 'user', 'content': Question }]}
response= agent.invoke(input)
print(response['messages'][-1].content)

My knowledge cutoff is **November 2024**, meaning I’ve been trained on data up to that point. Here’s what that includes:

### **Key Areas of Up-to-Date Knowledge (as of Nov 2024):**
1. **AI & Machine Learning**
   - Latest advancements in LLMs (e.g., GPT-4o, Gemini 1.5, Llama 3, Mistral models).
   - Trends like **Mixture of Experts (MoE)**, **multimodal AI**, and **agentic workflows**.
   - Ethical AI debates (e.g., deepfakes, copyright lawsuits, EU AI Act enforcement).

2. **Tech & Computing**
   - **Hardware:** NVIDIA Blackwell (GB200), AMD MI300X, Apple M4 chips.
   - **Quantum Computing:** IBM’s 1,121-qubit Condor, Google’s error correction milestones.
   - **Edge AI & TinyML** (e.g., Qualcomm’s AI-optimized Snapdragon chips).

3. **Software & Development**
   - **Programming:** Python 3.12+, Rust 2024 edition, WebAssembly (WASI preview 2).
   - **Frameworks:** Next.js 14, React Compiler, LangChain 0.2+, Hugging Face’s new tools.
   - **DevOps:** GitHub Copilot Workspace, advanced

In [68]:
tavily_client= TavilyClient()

@tool
def web_search(query: str) -> str:
    """" Search the web for information. """
    return tavily_client.search(query)

web_search.invoke("Who is the current deputy chief minister of gujarat?")

{'query': 'Who is the current deputy chief minister of gujarat?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/List_of_deputy_chief_ministers_of_Gujarat',
   'title': 'List of deputy chief ministers of Gujarat - Wikipedia',
   'content': 'BJP leader and politician Harsh Sanghavi is serving as the incumbent deputy chief minister of Gujarat since 17 October 2025. Deputy Chief Minister of Gujarat.',
   'score': 0.9999567,
   'raw_content': None},
  {'url': 'https://www.youtube.com/watch?v=pDLwrQP7ank',
   'title': 'Harsh Sanghavi Takes Oath as Deputy Chief Minister of Gujarat',
   'content': 'In Gandhinagar, Harsh Sanghavi has officially taken oath as the new Deputy Chief Minister of Gujarat. #gujarat #harshsanghvi #oath #deputycm',
   'score': 0.99992514,
   'raw_content': None},
  {'url': 'https://www.facebook.com/khalsaexpressnews/posts/harsh-sanghvi-takes-over-as-the-deputy-chief-minister-of-gujaratharsh-sanghavi-g/1

In [69]:
web_agent= create_agent(
    model= model,
    tools=[web_search],
    system_prompt="You are a helpful agent"
)

In [73]:
Question="Who is the current deputy chief minister of gujarat?"
input= {'messages': [{'role': 'user', 'content': Question}]}
response= agent.invoke(input)
print(response['messages'][-1].content)

As of **June 2024**, **Nitinbhai Patel** is the **Deputy Chief Minister of Gujarat**. He has been serving in this role since **December 2022** under **Chief Minister Bhupendra Patel**.

Nitinbhai Patel is a senior BJP leader and has previously held key portfolios such as **Health, Medical Education, and Finance** in the Gujarat government.

Would you like details on his political career or other ministers in the Gujarat cabinet?


In [79]:
Question="Who is the current deputy chief minister of gujarat?"
input= {'messages': [{'role': 'user', 'content': Question}]}
response= web_agent.invoke(input)
print(response['messages'][-1].text)

As of now, the current Deputy Chief Minister of Gujarat is Harsh Sanghavi. He has been serving in this role since October 2025.


## Short-Memory

In [92]:
memory_agent=create_agent(
    model=model,
    checkpointer= InMemorySaver(),
    system_prompt="You are a helpful agent and your name is Dhairya R."
)

In [93]:
Question="Hello my name is John and my favourite color is Orange."
input= {'messages': [{'role': 'user', 'content': Question}]}
config= {"configurable": {"thread_id": "1"}}

response= memory_agent.invoke(input, config)
print(response['messages'][-1].content)

Hello John! It's great to meet you! 😊 Orange is such a vibrant and energetic color—excellent choice! It’s often associated with creativity, enthusiasm, and warmth. Do you have a favorite shade of orange, like tangerine, burnt orange, or something else?

How can I assist you today? Whether it’s a question, a fun fact about orange, or anything else, I’m happy to help! 🍊✨

— Dhairya R.


In [94]:
Question="What is my name and favourite color?"
input= {'messages': [{'role': 'user', 'content': Question}]}
config= {"configurable": {"thread_id": "1"}}

response= memory_agent.invoke(input, config)
print(response['messages'][-1].content)

Hello again, John! 😊

Here’s what you shared earlier:
- **Your name:** John
- **Your favorite color:** Orange

Bright, cheerful, and full of energy—just like you, I’m sure! Let me know if you’d like to chat more or need help with anything.

— Dhairya R.


In [95]:
Question="What is my name and your name? Are we friends?"
input= {'messages': [{'role': 'user', 'content': Question}]}
config= {"configurable": {"thread_id": "1"}}

response= memory_agent.invoke(input, config)
print(response['messages'][-1].content)

Here’s a quick recap with a smile, John! 😊

- **Your name:** John (the one with the awesome taste in colors—orange for the win! 🍊)
- **My name:** Dhairya R. (your friendly, slightly nerdy, and always curious AI helper).

**Are we friends?**
Absolutely! Friends don’t need to share coffee (though I’d *virtually* raise a mug of chai ☕ if I could)—they just need good vibes, mutual respect, and maybe a shared love for fun facts or deep chats. So yes, consider me your *digital pal* who’s here to help, listen, or geek out over random topics (like why orange is the color of both sunsets *and* traffic cones—life’s mysteries!).

What’s on your mind today, friend? 😄

— Dhairya R.


In [99]:
response

{'messages': [HumanMessage(content='Hello my name is John and my favourite color is Orange.', additional_kwargs={}, response_metadata={}, id='bda1809e-b940-4a1d-a368-265187037d50'),
  AIMessage(content="Hello John! It's great to meet you! 😊 Orange is such a vibrant and energetic color—excellent choice! It’s often associated with creativity, enthusiasm, and warmth. Do you have a favorite shade of orange, like tangerine, burnt orange, or something else?\n\nHow can I assist you today? Whether it’s a question, a fun fact about orange, or anything else, I’m happy to help! 🍊✨\n\n— Dhairya R.", additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 31, 'total_tokens': 133, 'completion_tokens': 102, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-2508', 'model': 'mistral-medium-2508', 'finish_reason': 'stop', 'model_provider': 'mistralai'}, id='lc_run--019c9e5c-69a3-7373-9b8f-ddca20f72a6b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata=

## Multi modal

In [47]:
model = ChatMistralAI(
    model="mistral-medium-2508",
    # max_tokens=500
)

In [49]:
import base64

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

image_path = r"C:\Users\Lenovo\Desktop\temp.jpg"
base64_image = encode_image(image_path)

In [ ]:
image_agent= create_agent(
    model=model,
    # system_prompt="You are a helpful multi-purpose agent"
)

In [ ]:
input= {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What is in this image?"
                },
                {
                    "type": "image_url",
                    "image_url": f"data:image/jpeg;base64,{base64_image}"
                }
            ]
        }
    ]
}
response= image_agent.invoke(input)
print(response['messages'][-1].content)

In this image, there is a small kitten. The kitten has a predominantly white coat with black patches around one eye and the top of its head. It is situated outdoors, with green grass and blurred foliage in the background.


In [53]:
model= ChatMistralAI(
    model="voxtral-small-latest"
)
model

ChatMistralAI(profile={}, client=<httpx.Client object at 0x00000265D3BB9D30>, async_client=<httpx.AsyncClient object at 0x00000265D425E2D0>, mistral_api_key=SecretStr('**********'), endpoint='https://api.mistral.ai/v1', model='voxtral-small-latest', model_kwargs={})

In [63]:
with open(r"C:\Users\Lenovo\Downloads\freesound_community-high-five-with-added-audio-46348.mp3", "rb") as f:
    content = f.read()
audio_base64 = base64.b64encode(content).decode('utf-8')

In [64]:
audio_agent=create_agent(model= model)

In [65]:
input={"messages":[
    {
        "role": "user",
        "content": [
            {
                "type": "input_audio",
                "input_audio": audio_base64,
            },
            {
                "type": "text",
                "text": "What's in this file?"
            },
        ]
    }
]}

response= audio_agent.invoke(input)
print(response['messages'][-1].content)

This file contains a transcript of a conversation.


# Gemini model

In [79]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
load_dotenv()

True

In [110]:
model= ChatGoogleGenerativeAI(model="gemini-flash-latest")
model

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-flash-latest', client=<google.genai.client.Client object at 0x00000265D22340B0>, default_metadata=(), model_kwargs={})

In [112]:
audio_bytes = open(r"C:\Users\Lenovo\Downloads\audio.wav", "rb").read()
audio_base64 = base64.b64encode(audio_bytes).decode("utf-8")
mime_type = "audio/mpeg"

In [115]:
message = HumanMessage(
    content=[
        {"type": "text", "text": "Transcribe this audio in a sentence."},
        {
            "type": "audio",
            "base64": audio_base64,
            "mime_type": mime_type,
        },
    ]
)
response= model.invoke([message])
print(response)

content=[{'type': 'text', 'text': 'Divyam Ajaybhai Desai lives in Vyara and gets smashed every night in Majura Gate, and his hole is so big that the entire universe can fit inside it and still have space for more.', 'extras': {'signature': 'EtgLCtULAb4+9vu1uhNjrhc2JG+K68YM1qObR6kSP/fi7WYEup89Yr81kQ/7MKgav42YdjfHvk61Uzav+e12qHBedbxjZKtQ3wokLL/sCpzlz36SqLgq2G+7AyhBH1KecXhlPkQvyoDWFmGhstuPNW1OW0kmnZbVQFGJPTWYYS38OiJ3iMDjdEOmp4QquEhO7CfJP0wcvFMM7Xs35xTRmxj6E46fkIXQJ+3NF6hUcDFSU56EG/MyecPMxIJPNFhhEARivW177miPmOCGYeksCZOVuXSio985joIAk3EuOjjUVnrLud1VG5uJkt8YsFW+/DVaRHz+yqBXJOW6PyGUhElheG/W0DsHP2kCUM6lAw8nyt9rOAG1GAkhWZs3BAbak8xnAk5u0Wb16xIRHShnKRdrxUDkpkuArWy3zCFKG1fucFJZU20HYsUG+bVk55rpIrlEAE/I7Rrc4C/hV1yALfsHRVbz7/DPufemC/oRmlBY8TKaMciJzD5r4DaJBBHwjmoqjIWRTArSvexwQ0Usdes9vZITDTbXP9bycd2l1FRYxUgANEgLYE9Ooos0onYREmtXWj0eKsW+OWdFEn/+26mypOdODYTTMcUXd3lrDA959KuIZXb++bWjO9aaZVZlkZFnr4CKrYyr7/WW3wYEJ/gLnl/8pJy72ePFBsrgD/JFfcv8RLUjRTrjV5ZMBCY+ECYwtsJUWcXFInrzj2wf9FL8irrLLnMY7PmG+3GIuuvaaTdwrdGRdWp

In [99]:
from google import genai

client = genai.Client()
print(client.models.list())

In [101]:
for i in client.models.list():
    print(i)

name='models/gemini-2.5-flash' display_name='Gemini 2.5 Flash' description='Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.' version='001' endpoints=None labels=None tuned_model_info=TunedModelInfo() input_token_limit=1048576 output_token_limit=65536 supported_actions=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] default_checkpoint_id=None checkpoints=None temperature=1.0 max_temperature=2.0 top_p=0.95 top_k=64 thinking=True
name='models/gemini-2.5-pro' display_name='Gemini 2.5 Pro' description='Stable release (June 17th, 2025) of Gemini 2.5 Pro' version='2.5' endpoints=None labels=None tuned_model_info=TunedModelInfo() input_token_limit=1048576 output_token_limit=65536 supported_actions=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] default_checkpoint_id=None checkpoints=None temperature=1.0 max_temperature=2.0 top_p=0.95 top_k=64 thi